NOTE: This is an altered copy of a notebook provided by the organizers of the Current Topics in Digital Philology course.

### **Current Topics in Digital Philology 5LN720/5LN721**

# **Lab 5: Assignment on Digital Methods in Literary Analysis**



## **1. Get started**

First, we import the Gutenberg corpus and some packages and tools that might be useful.

In [1]:
import nltk

from nltk.corpus import gutenberg  # The Gutenberg corpus
from nltk.corpus import (
    stopwords,
)  # Stop words = a set of high frequent words in a language (e.g. “the”, “is”, “and”) that you might want to filter out
from nltk import (
    word_tokenize,
    sent_tokenize,
)  # NLTK package for tokenizing words or sentences
from nltk import WordNetLemmatizer  # NLTK package for lemmatizing words

# import string
# from collections import OrderedDict
# import matplotlib.pyplot as plt
# from collections import Counter
# import re

nltk.download("gutenberg")
nltk.download("stopwords")
nltk.download("punkt")
nltk.download("averaged_perceptron_tagger")
nltk.download("universal_tagset")
nltk.download("wordnet")
nltk.download("punkt_tab")
nltk.download("averaged_perceptron_tagger_eng")

print("Done!")

Done!


[nltk_data] Downloading package gutenberg to C:\Users\Daan
[nltk_data]     Brugmans\AppData\Roaming\nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Daan
[nltk_data]     Brugmans\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to C:\Users\Daan
[nltk_data]     Brugmans\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Daan Brugmans\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package universal_tagset to C:\Users\Daan
[nltk_data]     Brugmans\AppData\Roaming\nltk_data...
[nltk_data]   Package universal_tagset is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\Daan
[nltk_data]     Brugmans\AppData\Roaming\nlt

## **3. Data Selection**

In [2]:
books = [
    "carroll-alice.txt",
    "austen-sense.txt",
]  # <------------------------------------------------  1. CHANGE TO THE BOOK(S) OF YOUR COICE!

data = []
data_sents = []
for book in books:
    data.append(gutenberg.words(book))  # data as tokens
    data_sents.append(gutenberg.sents(book))  # data as sentences, used for pos tagging

print("Example of how the data as tokens looks like:")
print((" ".join(data_sents[0][0]).replace("[ ", "").replace(" ]", "")))

train_set = [
    "austen-emma.txt",
    "chesterton-ball.txt",
    "shakespeare-caesar.txt",
    "austen-persuasion.txt",
    "chesterton-brown.txt",
    "shakespeare-hamlet.txt",
]

# val_set = [
#     "austen-persuasion.txt",
#     "chesterton-brown.txt",
#     "shakespeare-hamlet.txt",
# ]

test_set = [
    "austen-sense.txt",
    "chesterton-thursday.txt",
    "shakespeare-macbeth.txt",
]

Example of how the data as tokens looks like:
Alice ' s Adventures in Wonderland by Lewis Carroll 1865


# **Custom Code Intermission**

In [3]:
import os
from copy import deepcopy
import json

import pandas as pd
import matplotlib.pyplot as plt

from featurizer import Featurizer
from classifier import Classifier


RERUN_FEATURES = True


def main():
    featurizer = Featurizer()

    full_train_set = get_features(
        featurizer,
        RERUN_FEATURES,
        feature_filename="data/train_features.csv",
        filename=train_set,
    )
    # full_dev_set = get_features(
    #     featurizer,
    #     RERUN_FEATURES,
    #     feature_filename="data/dev_features.csv",
    #     filename=val_set,
    # )
    full_test_set = get_features(
        featurizer,
        RERUN_FEATURES,
        feature_filename="data/test_features.csv",
        filename=test_set,
    )

    print(full_train_set["author"].value_counts())
    print("skibidi")
    print(full_test_set["author"].value_counts())

    # exit()
    print("Running classifier...")
    classifier = Classifier()

    colnames = [col for col in full_train_set.columns if col.startswith("f_")]

    train_x = full_train_set[colnames].to_numpy()
    train_y = full_train_set["author"].to_numpy()
    classifier.fit(train_x, train_y)

    print("Evaluating classifier...")
    # train_report = classifier.evaluate(train_x, train_y)

    # dev_x = full_dev_set[colnames].to_numpy()
    # dev_y = full_dev_set["author"].to_numpy()
    # dev_report = classifier.evaluate(dev_x, dev_y)

    test_x = full_test_set[colnames].to_numpy()
    test_y = full_test_set["author"].to_numpy()
    test_report = classifier.evaluate(test_x, test_y)

    report = {
        # "train_report": train_report,
        # "dev_report": dev_report,
        "test_report": test_report,
    }
    with open("classifier_results.json", "w") as outfile:
        json.dump(report, outfile, indent=4)

    print("Doing ablation study")
    # perform_ablation_study(colnames, full_train_set, full_dev_set, full_test_set)
    perform_ablation_study(colnames, full_train_set, full_test_set)
    # perform_ablation_study(
    #     colnames, full_train_set, full_dev_set, full_test_set, reverse=True
    # )
    # perform_ablation_study(
    #     colnames, full_train_set, full_dev_set, full_test_set, groups=True
    # )
    # perform_ablation_study(
    #     colnames, full_train_set, full_dev_set, full_test_set, groups=True, reverse=True
    # )


def get_features(
    featurizer: Featurizer, rerun_features: bool, feature_filename: str, filename: str
) -> pd.DataFrame:
    """Get the features for texts and return a dataframe with the text and the features

    Args:
        featurizer (Featurizer): Object that performs featurization
        rerun_features (bool): If the features should be re-calculated
        feature_filename (str): Filename for the calculated features
        filename (str): Filename containing the texts and authors

    Returns:
        pd.Dataframe: All texts, their features and their author
    """
    # data_set = pd.read_csv(filename, index_col=0)
    # data_set.index.names = ["index"]

    if rerun_features or not os.path.exists(feature_filename):
        print("Calculating features...")
        features = featurizer.featurize(filename)
        features.to_csv(feature_filename)
        summary = features.describe(include="all")
        summary.to_csv(f"{feature_filename.split('.')[0].split('_')[0]}_summary.csv")
    else:
        print("Loading features...")
        features = pd.read_csv(feature_filename)

    return features


def perform_ablation_study(
    colnames: list,
    full_train_set: pd.DataFrame,
    # full_dev_set: pd.DataFrame,
    full_test_set: pd.DataFrame,
    groups: bool = False,
    reverse: bool = False,
) -> None:
    """Perform an ablation study

    Args:
        colnames (list): Names of the feature columns in the datasets
        full_train_set (pd.DataFrame): Train set
        full_dev_set (pd.DataFrame): Dev set
        full_test_set (pd.DataFrame): Test set
        groups (bool, optional): If the features should be grouped. Defaults to False.
        reverse (bool, optional): When true, the classifiers are trained for each individual feature(group),
            insead of leaving out that feature(group). Defaults to False.
    """
    # train_results = {}
    # dev_results = {}
    test_results = {}
    train_y = full_train_set["author"].to_numpy()
    # dev_y = full_dev_set["author"].to_numpy()
    test_y = full_test_set["author"].to_numpy()
    features = ["d", "c", "p", "g", "i", "o"] if groups else colnames
    for feature in features:
        if groups:
            if reverse:
                ablation_colnames = [
                    colname for colname in colnames if colname[2] == feature
                ]
            else:
                ablation_colnames = [
                    colname for colname in colnames if colname[2] != feature
                ]
        else:
            if reverse:
                ablation_colnames = [feature]
            else:
                ablation_colnames = deepcopy(colnames)
                ablation_colnames.remove(feature)

        train_x = full_train_set[ablation_colnames].to_numpy()
        # dev_x = full_dev_set[ablation_colnames].to_numpy()
        test_x = full_test_set[ablation_colnames].to_numpy()

        classifier = Classifier()
        classifier.fit(train_x=train_x, train_y=train_y)

        # train_results[feature] = classifier.get_f1_score(eval_x=train_x, eval_y=train_y)
        # dev_results[feature] = classifier.get_f1_score(eval_x=dev_x, eval_y=dev_y)
        test_results[feature] = classifier.get_f1_score(eval_x=test_x, eval_y=test_y)

    settings_str = f"{'_groups' if groups else ''}{'_reverse' if reverse else ''}"
    results = {
        # "train_results": train_results,
        # "dev_results": dev_results,
        "test_results": test_results,
    }
    with open(f"results{settings_str}.json", "w") as outfile:
        json.dump(results, outfile, indent=4)

    # plot_ablation_results(train_results, name="Train", settings_str=settings_str)
    # plot_ablation_results(dev_results, name="Dev", settings_str=settings_str)
    plot_ablation_results(test_results, name="Test", settings_str=settings_str)


def plot_ablation_results(
    ablation_results: dict, name: str = "", settings_str: str = ""
) -> None:
    """Plot the results of an ablation study

    Args:
        ablation_results (dict): Results of the ablation study. Keys are feature(group) names. Values are F1 scores
        name (str, optional): Name of the dataset the classifiers were evaluated on. Defaults to "".
        settings_str (str, optional): Settings of the ablation study. Defaults to "".
    """

    if "groups" in settings_str:
        x_ticks = [
            "Default Counts",
            "Complexity",
            "POS tags",
            "Grammar/spelling",
            "Punctuation",
            "Other",
        ]
        rotation = 45
        bottom = 0.3
        figsize = (7, 5)
    else:
        x_ticks = [i[4:] for i in ablation_results.keys()]
        rotation = 90
        bottom = 0.45
        figsize = (10, 5)

    plt.figure(figsize=figsize)
    plt.bar(x_ticks, ablation_results.values())
    plt.xticks(rotation=rotation)
    plt.xlabel("Missing feature")
    plt.ylabel("F1 score")
    plt.title(
        f"{name} - Ablation study for authorship attribution ({' '.join(settings_str.split('_')).strip()})"
    )
    plt.subplots_adjust(top=0.9, bottom=bottom)
    plt.savefig(f"ablation_plot_{name.lower()}{settings_str}.png")
    plt.close()


main()

c:\Users\Daan Brugmans\Projects\uu-current-topics-in-digital-philology-25-26\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package punkt to C:\Users\Daan
[nltk_data]     Brugmans\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Daan Brugmans\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package universal_tagset to C:\Users\Daan
[nltk_data]     Brugmans\AppData\Roaming\nltk_data...
[nltk_data]   Package universal_tagset is already up-to-date!
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1606.54it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight

Calculating features...


100%|██████████| 6/6 [00:15<00:00,  2.65s/it]


Calculating features...


100%|██████████| 3/3 [00:06<00:00,  2.31s/it]


author
austen         2301
chesterton     1718
shakespeare    1055
Name: count, dtype: int64
skibidi
author
austen         1000
chesterton      749
shakespeare     382
Name: count, dtype: int64
Running classifier...
Evaluating classifier...
Accuracy: 0.8343500703894885
F1 score: 0.8347902634526742
Classification report:               precision    recall  f1-score   support

      austen       0.85      0.80      0.82      1000
  chesterton       0.75      0.81      0.78       749
 shakespeare       0.97      0.98      0.98       382

    accuracy                           0.83      2131
   macro avg       0.86      0.86      0.86      2131
weighted avg       0.84      0.83      0.83      2131

Doing ablation study


c:\Users\Daan Brugmans\Projects\uu-current-topics-in-digital-philology-25-26\env\Lib\site-packages\xgboost\core.py:751: UserWarning: [14:42:02] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


KeyboardInterrupt: 